# Chapter 14: Advertiser Measurement Analytics - Code Validation

This notebook tests the code examples from Chapter 14 to ensure they are production-quality and executable.

## Table of Contents
1. Bid Landscape Estimation via Survival Analysis (Code Listing 14.1)
2. Media Mix Model with Adstock and Saturation (Code Listing 14.2)

## 1. Code Listing 14.1 — Bid Landscape Estimation via Survival Analysis (Section 2.2)

Tests for `kaplan_meier_win_rate` and `expected_cost_curve` from Code Listing 14.1. We verify:
- **1a.** All-uncensored data (all wins) recovers empirical CDF
- **1b.** Win rate is monotonically non-decreasing
- **1c.** Win rate = 0 at bid 0, approaches 1.0 at high bids
- **1d.** Censoring shifts the estimated win-rate curve (lost auctions withhold information)
- **1e.** Expected cost ≤ bid (second-price property)
- **1f.** Known uniform distribution: KM approximates true CDF

In [1]:
# --- Load Code Listing 14.1: Bid Landscape Estimation ---
import numpy as np
from typing import NamedTuple


class AuctionRecord(NamedTuple):
    """A single auction observation for bid landscape estimation."""
    price: float     # clearing price (if won) or advertiser's bid (if lost)
    won: bool        # True = price observed; False = right-censored at bid


def kaplan_meier_win_rate(
    records: list[AuctionRecord], bid_grid: np.ndarray
) -> np.ndarray:
    """Estimate win-rate curve W(b) via the Kaplan-Meier survival estimator.

    Maps clearing price -> survival time, winning -> uncensored event.
    Won auctions: event observed at clearing price.
    Lost auctions: right-censored at the advertiser's bid.

    Returns win_rates: array of P(win) at each bid in bid_grid.
    """
    prices = np.array([r.price for r in records])
    observed = np.array([r.won for r in records])

    order = np.argsort(prices)
    prices = prices[order]
    observed = observed[order]

    # Build Kaplan-Meier survival curve: S(b) = P(clearing_price > b)
    unique_prices = np.unique(prices)
    surv = 1.0
    km_prices = [0.0]
    km_surv = [1.0]

    for p in unique_prices:
        at_risk = int(np.sum(prices >= p))
        events = int(np.sum((prices == p) & observed))
        if at_risk > 0 and events > 0:
            surv *= 1.0 - events / at_risk
        km_prices.append(p)
        km_surv.append(surv)

    km_prices = np.array(km_prices)
    km_surv = np.array(km_surv)

    # Win rate W(b) = 1 - S(b), interpolated to bid grid
    win_rates = np.zeros(len(bid_grid))
    for i, b in enumerate(bid_grid):
        idx = np.searchsorted(km_prices, b, side="right") - 1
        idx = np.clip(idx, 0, len(km_surv) - 1)
        win_rates[i] = 1.0 - km_surv[idx]

    return win_rates


def expected_cost_curve(
    records: list[AuctionRecord], bid_grid: np.ndarray
) -> np.ndarray:
    """Estimate E[cost | win] at each bid level.

    In a second-price auction, cost = clearing price. At bid b,
    E[cost | win] = E[clearing_price | clearing_price <= b].
    """
    prices = np.array([r.price for r in records])
    observed = np.array([r.won for r in records])

    costs = np.zeros(len(bid_grid))
    for i, b in enumerate(bid_grid):
        mask = observed & (prices <= b)
        if np.any(mask):
            costs[i] = np.mean(prices[mask])

    return costs


print("\u2713 Code Listing 14.1 loaded")

✓ Code Listing 14.1 loaded


In [2]:
# --- Test 1a: All-uncensored data recovers empirical CDF ---
# When all auctions are won (no censoring), KM should match the empirical CDF.
np.random.seed(42)
clearing_prices = np.random.uniform(0.5, 5.0, size=500)
records_all_won = [AuctionRecord(price=p, won=True) for p in clearing_prices]

bid_grid = np.linspace(0.0, 6.0, 100)
win_rates = kaplan_meier_win_rate(records_all_won, bid_grid)

# Compare with empirical CDF
for b, w in zip(bid_grid[::20], win_rates[::20]):
    empirical = np.mean(clearing_prices <= b)
    assert abs(w - empirical) < 0.02, f"KM mismatch at bid={b:.1f}: {w:.3f} vs {empirical:.3f}"

print("1a: KM win rates match empirical CDF (max error < 0.02)")
print("\u2713 All-uncensored data correctly recovers empirical CDF")

1a: KM win rates match empirical CDF (max error < 0.02)
✓ All-uncensored data correctly recovers empirical CDF


In [3]:
# --- Test 1b: Win rate is monotonically non-decreasing ---
diffs = np.diff(win_rates)
assert np.all(diffs >= -1e-10), f"Win rate decreased: min diff = {diffs.min():.6f}"
print("1b: Win-rate curve is monotonically non-decreasing")
print("\u2713 Monotonicity property holds")

# --- Test 1c: Win rate = 0 at bid 0, approaches 1.0 at high bid ---
assert win_rates[0] == 0.0, f"Expected 0 at bid=0, got {win_rates[0]}"
assert win_rates[-1] >= 0.99, f"Expected ~1.0 at bid=6.0, got {win_rates[-1]:.3f}"
print(f"\n1c: W(0) = {win_rates[0]:.1f}, W(6.0) = {win_rates[-1]:.3f}")
print("\u2713 Boundary conditions hold (0 at zero bid, ~1.0 at high bid)")

1b: Win-rate curve is monotonically non-decreasing
✓ Monotonicity property holds

1c: W(0) = 0.0, W(6.0) = 1.000
✓ Boundary conditions hold (0 at zero bid, ~1.0 at high bid)


In [4]:
# --- Test 1d: Censoring shifts the estimated win-rate curve ---
# Create mixed data: some won (observe clearing price), some lost (censored).
# Advertiser bid = 3.0. Auctions with clearing price <= 3.0 are won;
# auctions with clearing price > 3.0 are lost (censored at 3.0).
np.random.seed(7)
true_prices = np.random.uniform(0.5, 5.0, size=1000)
advertiser_bid = 3.0

records_mixed = []
for p in true_prices:
    if p <= advertiser_bid:
        records_mixed.append(AuctionRecord(price=p, won=True))
    else:
        records_mixed.append(AuctionRecord(price=advertiser_bid, won=False))

won_count = sum(1 for r in records_mixed if r.won)
lost_count = sum(1 for r in records_mixed if not r.won)

win_rates_mixed = kaplan_meier_win_rate(records_mixed, bid_grid)

# KM should still produce a valid curve
assert np.all(np.diff(win_rates_mixed) >= -1e-10), "Mixed data broke monotonicity"

# At bid = 3.0, the win rate should approximately match the true fraction won
true_win_frac = np.mean(true_prices <= advertiser_bid)
idx_3 = np.argmin(np.abs(bid_grid - advertiser_bid))
km_at_3 = win_rates_mixed[idx_3]
assert abs(km_at_3 - true_win_frac) < 0.05, (
    f"W(3.0) = {km_at_3:.3f}, expected ~{true_win_frac:.3f}"
)

print(f"1d: {won_count} won, {lost_count} lost (censored at bid={advertiser_bid})")
print(f"    KM W(3.0) = {km_at_3:.3f}, true fraction won = {true_win_frac:.3f}")
print("\u2713 Censored data produces valid win-rate curve consistent with true distribution")

1d: 571 won, 429 lost (censored at bid=3.0)
    KM W(3.0) = 0.566, true fraction won = 0.571
✓ Censored data produces valid win-rate curve consistent with true distribution


In [5]:
# --- Test 1e: Expected cost <= bid (second-price property) ---
costs = expected_cost_curve(records_all_won, bid_grid)

for b, c in zip(bid_grid, costs):
    if c > 0:  # skip zero-cost entries (no won auctions below this bid)
        assert c <= b + 1e-6, f"Cost {c:.3f} > bid {b:.3f}"

print("1e: Expected cost never exceeds bid at any level")
print("\u2713 Second-price property holds: E[cost | win] \u2264 bid")

1e: Expected cost never exceeds bid at any level
✓ Second-price property holds: E[cost | win] ≤ bid


In [6]:
# --- Test 1f: Known Uniform(0,5) distribution: KM approximates true CDF ---
np.random.seed(123)
n_auctions = 5000
true_clearing = np.random.uniform(0.0, 5.0, size=n_auctions)

# Advertiser bids uniformly in [1, 4] — creates realistic censoring pattern
advertiser_bids = np.random.uniform(1.0, 4.0, size=n_auctions)

records_uniform = []
for z, b in zip(true_clearing, advertiser_bids):
    if z <= b:
        records_uniform.append(AuctionRecord(price=z, won=True))
    else:
        records_uniform.append(AuctionRecord(price=b, won=False))

fine_grid = np.linspace(0.0, 5.0, 200)
km_win = kaplan_meier_win_rate(records_uniform, fine_grid)
true_cdf = fine_grid / 5.0  # Uniform(0,5) CDF

# Check accuracy in the well-observed range [0.5, 3.5]
mask_range = (fine_grid >= 0.5) & (fine_grid <= 3.5)
max_error = np.max(np.abs(km_win[mask_range] - true_cdf[mask_range]))
assert max_error < 0.05, f"Max KM error = {max_error:.4f} (expected < 0.05)"

print(f"1f: KM vs true Uniform(0,5) CDF — max error in [0.5, 3.5] = {max_error:.4f}")
print("\u2713 KM accurately recovers known distribution from censored data")

1f: KM vs true Uniform(0,5) CDF — max error in [0.5, 3.5] = 0.0107
✓ KM accurately recovers known distribution from censored data


## 2. Code Listing 14.2 — Media Mix Model with Adstock and Saturation (Section 3.1)

Tests for `geometric_adstock`, `hill_saturation`, and `fit_mmm` from Code Listing 14.2. We verify:
- **2a.** Adstock with decay=0 produces no carry-over (output = input)
- **2b.** Adstock with decay > 0 increases values for t > 0
- **2c.** Hill saturation at x = half_max returns 0.5
- **2d.** Hill saturation is monotonically increasing
- **2e.** MMM fit recovers known channel contributions from synthetic data
- **2f.** MMM intercept approximates true baseline

In [7]:
# --- Load Code Listing 14.2: Media Mix Model ---
import numpy as np


def geometric_adstock(spend: np.ndarray, decay: float) -> np.ndarray:
    """Geometric adstock transformation (Equation 14.9).

    adstock_t = spend_t + decay * adstock_{t-1}

    Captures the carry-over effect of advertising spend
    into subsequent time periods.
    """
    adstock = np.zeros_like(spend, dtype=float)
    adstock[0] = spend[0]
    for t in range(1, len(spend)):
        adstock[t] = spend[t] + decay * adstock[t - 1]
    return adstock


def hill_saturation(x: np.ndarray, half_max: float, slope: float) -> np.ndarray:
    """Hill saturation function for diminishing returns (Equation 14.10).

    f(x) = x^slope / (half_max^slope + x^slope)

    Returns values in [0, 1]. half_max is the spend level at which
    the response reaches 50% of its maximum.
    """
    return np.power(x, slope) / (np.power(half_max, slope) + np.power(x, slope))


def fit_mmm(
    spend_by_channel: np.ndarray,   # (T, C): weekly spend per channel
    sales: np.ndarray,               # (T,): total sales per week
    decay_rates: np.ndarray,         # (C,): adstock decay per channel
    half_maxes: np.ndarray,          # (C,): saturation half-max per channel
    slopes: np.ndarray,              # (C,): saturation slope per channel
) -> tuple[np.ndarray, float]:
    """Fit MMM channel coefficients via OLS.

    Given fixed transformation parameters (decay_rates, half_maxes, slopes),
    solves for the linear coefficients (betas) and intercept in closed form.
    In production, the transformation parameters are jointly optimized via
    Bayesian MCMC (Section 3.2) or gradient-based methods.

    Returns:
        betas:     channel contribution coefficients, shape (C,)
        intercept: baseline sales (organic conversions without ad spend)
    """
    T, C = spend_by_channel.shape

    X = np.zeros((T, C))
    for c in range(C):
        adstocked = geometric_adstock(spend_by_channel[:, c], decay_rates[c])
        X[:, c] = hill_saturation(adstocked, half_maxes[c], slopes[c])

    # OLS: sales = X @ betas + intercept
    X_aug = np.column_stack([X, np.ones(T)])
    coeffs, _, _, _ = np.linalg.lstsq(X_aug, sales, rcond=None)

    betas = coeffs[:C]
    intercept = coeffs[C]

    return betas, intercept


print("\u2713 Code Listing 14.2 loaded")

✓ Code Listing 14.2 loaded


In [8]:
# --- Test 2a: Adstock with decay=0 produces no carry-over ---
spend = np.array([100.0, 0.0, 0.0, 50.0, 0.0])
adstocked_0 = geometric_adstock(spend, decay=0.0)
np.testing.assert_array_equal(adstocked_0, spend)
print("2a: Adstock(decay=0) = input (no carry-over)")
print("\u2713 Zero decay produces identity transform")

# --- Test 2b: Adstock with decay > 0 increases values for t > 0 ---
adstocked_05 = geometric_adstock(spend, decay=0.5)
assert adstocked_05[0] == spend[0], "First element should equal input"
# After the $100 pulse, subsequent periods should carry over
assert adstocked_05[1] == 0.5 * 100.0, f"Expected 50.0, got {adstocked_05[1]}"
assert adstocked_05[2] == 0.5 * 50.0, f"Expected 25.0, got {adstocked_05[2]}"
# After the $50 spend at t=3, should include carry-over from previous
assert adstocked_05[3] == 50.0 + 0.5 * 25.0, f"Expected 62.5, got {adstocked_05[3]}"

print(f"\n2b: Adstock(decay=0.5) = {adstocked_05}")
print("\u2713 Positive decay produces correct carry-over")

2a: Adstock(decay=0) = input (no carry-over)
✓ Zero decay produces identity transform

2b: Adstock(decay=0.5) = [100.    50.    25.    62.5   31.25]
✓ Positive decay produces correct carry-over


In [9]:
# --- Test 2c: Hill saturation at x = half_max returns 0.5 ---
for half_max in [10.0, 100.0, 1000.0]:
    for slope in [0.5, 1.0, 2.0, 3.0]:
        val = hill_saturation(np.array([half_max]), half_max, slope)[0]
        assert abs(val - 0.5) < 1e-10, (
            f"Hill(K={half_max}, S={slope}) at x=K = {val:.6f}, expected 0.5"
        )

print("2c: Hill(x=K) = 0.5 for all tested (K, S) combinations")
print("\u2713 Half-saturation property holds")

# --- Test 2d: Hill saturation is monotonically increasing ---
x_grid = np.linspace(0.01, 200.0, 500)
for slope in [0.5, 1.0, 2.0]:
    sat = hill_saturation(x_grid, half_max=50.0, slope=slope)
    diffs = np.diff(sat)
    assert np.all(diffs > 0), f"Hill not monotonic for slope={slope}"

print("\n2d: Hill saturation is strictly monotonically increasing for all tested slopes")
print("\u2713 Monotonicity property holds")

2c: Hill(x=K) = 0.5 for all tested (K, S) combinations
✓ Half-saturation property holds

2d: Hill saturation is strictly monotonically increasing for all tested slopes
✓ Monotonicity property holds


In [10]:
# --- Test 2e: MMM fit recovers known channel contributions ---
# Generate synthetic data from known parameters
np.random.seed(99)
T = 52  # 52 weeks
C = 3   # 3 channels: search, display, social

# True parameters
true_betas = np.array([500.0, 300.0, 100.0])  # search > display > social
true_intercept = 1000.0  # baseline organic sales
true_decays = np.array([0.2, 0.6, 0.4])
true_half_maxes = np.array([50.0, 30.0, 20.0])
true_slopes = np.array([1.5, 1.0, 2.0])

# Generate spend data with some variation
spend = np.zeros((T, C))
spend[:, 0] = np.random.uniform(20, 80, T)   # search
spend[:, 1] = np.random.uniform(10, 50, T)   # display
spend[:, 2] = np.random.uniform(5, 30, T)    # social

# Generate sales from the true model (no noise for clean test)
sales = np.full(T, true_intercept)
for c in range(C):
    adstocked = geometric_adstock(spend[:, c], true_decays[c])
    saturated = hill_saturation(adstocked, true_half_maxes[c], true_slopes[c])
    sales += true_betas[c] * saturated

# Fit MMM with the true transformation parameters
betas, intercept = fit_mmm(spend, sales, true_decays, true_half_maxes, true_slopes)

print("2e: MMM fit with known transformation parameters:")
for i, name in enumerate(["search", "display", "social"]):
    print(f"    {name}: beta = {betas[i]:.1f} (true = {true_betas[i]:.1f})")
print(f"    intercept = {intercept:.1f} (true = {true_intercept:.1f})")

# Verify recovery
for i in range(C):
    assert abs(betas[i] - true_betas[i]) < 1.0, (
        f"Channel {i}: beta = {betas[i]:.2f}, expected {true_betas[i]:.2f}"
    )
assert abs(intercept - true_intercept) < 1.0, (
    f"Intercept = {intercept:.2f}, expected {true_intercept:.2f}"
)

print("\u2713 MMM correctly recovers true channel coefficients and intercept")

2e: MMM fit with known transformation parameters:
    search: beta = 500.0 (true = 500.0)
    display: beta = 300.0 (true = 300.0)
    social: beta = 100.0 (true = 100.0)
    intercept = 1000.0 (true = 1000.0)
✓ MMM correctly recovers true channel coefficients and intercept


In [11]:
# --- Test 2f: MMM intercept approximates true baseline with noise ---
np.random.seed(42)
noise = np.random.normal(0, 20, T)  # small Gaussian noise
sales_noisy = sales + noise

betas_n, intercept_n = fit_mmm(
    spend, sales_noisy, true_decays, true_half_maxes, true_slopes
)

# With noise, expect approximate recovery
print(f"2f: Noisy MMM fit (\u03c3=20):")
for i, name in enumerate(["search", "display", "social"]):
    pct_err = abs(betas_n[i] - true_betas[i]) / true_betas[i] * 100
    print(f"    {name}: beta = {betas_n[i]:.1f} (true = {true_betas[i]:.1f}, err = {pct_err:.1f}%)")
pct_err_int = abs(intercept_n - true_intercept) / true_intercept * 100
print(f"    intercept = {intercept_n:.1f} (true = {true_intercept:.1f}, err = {pct_err_int:.1f}%)")

# Channel ordering should be preserved (search > display > social)
assert betas_n[0] > betas_n[1] > betas_n[2], (
    f"Channel ordering lost: {betas_n}"
)
print("\u2713 MMM preserves channel ordering under noise (search > display > social)")

2f: Noisy MMM fit (σ=20):
    search: beta = 464.9 (true = 500.0, err = 7.0%)
    display: beta = 304.9 (true = 300.0, err = 1.6%)
    social: beta = 64.5 (true = 100.0, err = 35.5%)
    intercept = 1035.3 (true = 1000.0, err = 3.5%)
✓ MMM preserves channel ordering under noise (search > display > social)


## Summary

All tests validate the core behaviors of Code Listings 14.1 and 14.2:

| Test | Behavior Verified |
|------|-------------------|
| 1a. Empirical CDF | All-uncensored KM matches empirical CDF |
| 1b. Monotonicity | Win-rate curve is non-decreasing |
| 1c. Boundary conditions | W(0)=0, W(high)≈1.0 |
| 1d. Censoring | Right-censored data produces valid, consistent curve |
| 1e. Second-price property | Expected cost never exceeds bid |
| 1f. Known distribution | KM recovers Uniform(0,5) CDF from censored data |
| 2a. Zero decay | Adstock identity when decay=0 |
| 2b. Carry-over | Positive decay produces correct adstock accumulation |
| 2c. Half-saturation | Hill(x=K) = 0.5 for all parameter combinations |
| 2d. Monotonicity | Hill function strictly increasing |
| 2e. Recovery | MMM recovers true betas and intercept from noiseless data |
| 2f. Robustness | MMM preserves channel ordering under noise |